In [3]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. HÀM MATCH ĐÚNG DÒNG GOLD SET VỚI FULL SET
#    Không dùng id đơn lẻ vì id trong gold set là id cục bộ.
# ============================================================

def norm_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFC", str(x)).lower()
    x = re.sub(r"\s+", " ", x).strip()
    return x


def ensure_match_columns(df):
    df = df.copy()

    # Nếu thiếu cột thì tạo rỗng để code vẫn chạy được.
    for col in ["topic", "video_id", "source_item_id", "created_at", "text", "clean_text"]:
        if col not in df.columns:
            df[col] = ""

    df["__topic_key"] = df["topic"].apply(norm_text)
    df["__video_id_key"] = df["video_id"].apply(norm_text)
    df["__source_item_id_key"] = df["source_item_id"].apply(norm_text)
    df["__created_at_key"] = df["created_at"].apply(norm_text)
    df["__text_key"] = df["text"].apply(norm_text)
    df["__clean_text_key"] = df["clean_text"].apply(norm_text)

    return df


def build_key(row, fields):
    return "||".join(str(row.get(f"__{field}_key", "")) for field in fields)


def is_valid_key(key, fields):
    parts = key.split("||")
    if len(parts) != len(fields):
        return False

    # topic luôn bắt buộc.
    if "topic" in fields and parts[fields.index("topic")] == "":
        return False

    # Nếu dùng text / clean_text thì không được rỗng.
    for field in ["text", "clean_text"]:
        if field in fields and parts[fields.index(field)] == "":
            return False

    # Nếu dùng các khóa mạnh này thì không được rỗng để tránh match nhầm hàng loạt.
    for field in ["video_id", "source_item_id", "created_at"]:
        if field in fields and parts[fields.index(field)] == "":
            return False

    return True


def make_full_key_map(full_work, fields):
    tmp = full_work[["__full_index"] + [f"__{field}_key" for field in fields]].copy()
    tmp["__match_key"] = tmp.apply(lambda row: build_key(row, fields), axis=1)
    tmp = tmp[tmp["__match_key"].apply(lambda key: is_valid_key(key, fields))]

    return tmp.groupby("__match_key")["__full_index"].apply(list).to_dict()


def match_gold_to_full(gold, full):
    """
    Trả về dataframe 1 dòng / 1 gold row.
    Nếu match nhiều duplicate trong full set thì lấy full row đầu tiên để benchmark.
    """
    gold_work = ensure_match_columns(gold)
    full_work = ensure_match_columns(full)

    gold_work["__gold_index"] = gold_work.index
    full_work["__full_index"] = full_work.index

    # Thứ tự match: khóa mạnh trước, fallback unique sau.
    match_stages = [
        ("topic+video_id+created_at+text", ["topic", "video_id", "created_at", "text"], False),
        ("topic+source_item_id+created_at+text", ["topic", "source_item_id", "created_at", "text"], False),
        ("topic+video_id+text", ["topic", "video_id", "text"], False),
        ("topic+source_item_id+text", ["topic", "source_item_id", "text"], False),
        ("topic+created_at+text", ["topic", "created_at", "text"], False),

        # Fallback khi text trong Excel bị biến dạng, chỉ dùng nếu unique.
        ("topic+video_id+created_at_unique", ["topic", "video_id", "created_at"], True),
        ("topic+source_item_id+created_at_unique", ["topic", "source_item_id", "created_at"], True),

        # Fallback cuối: chỉ dùng nếu unique trong full.
        ("topic+text_unique", ["topic", "text"], True),
        ("topic+clean_text_unique", ["topic", "clean_text"], True),
    ]

    # Chuẩn bị map theo từng stage.
    stage_maps = []
    for method, fields, require_unique in match_stages:
        full_map = make_full_key_map(full_work, fields)
        stage_maps.append((method, fields, require_unique, full_map))

    rows = []

    for _, gold_row in gold_work.iterrows():
        gold_idx = int(gold_row["__gold_index"])
        matched = False

        for method, fields, require_unique, full_map in stage_maps:
            key = build_key(gold_row, fields)
            if not is_valid_key(key, fields):
                continue

            candidates = full_map.get(key, [])
            if not candidates:
                continue

            candidates = sorted(set(int(x) for x in candidates))

            if require_unique and len(candidates) != 1:
                continue

            full_idx = candidates[0]

            rows.append({
                "gold_index": gold_idx,
                "matched_full_index": full_idx,
                "matched_full_id": full.loc[full_idx, "id"] if "id" in full.columns else pd.NA,
                "match_method": method,
                "matched_full_count": len(candidates),
            })

            matched = True
            break

        if not matched:
            rows.append({
                "gold_index": gold_idx,
                "matched_full_index": pd.NA,
                "matched_full_id": pd.NA,
                "match_method": "unmatched",
                "matched_full_count": 0,
            })

    return pd.DataFrame(rows)


In [4]:
# ============================================================
# 2. ĐỌC FILE VÀ MATCH GOLD SET VỚI FULL SET
# ============================================================

gold_path = "human_gold_500_final.csv"
full_path = "labeled_data_oss.csv"

gold = pd.read_csv(gold_path)
full = pd.read_csv(full_path)

print("Gold shape:", gold.shape)
print("Full shape:", full.shape)

# Đảm bảo các nhãn đã là số 0/1/2 như bạn đã chuẩn hóa.
gold["final_is_relevant"] = gold["final_is_relevant"].astype(int)
gold["final_manual_label"] = gold["final_manual_label"].astype(int)
full["is_relevant"] = full["is_relevant"].astype(int)
full["manual_label"] = full["manual_label"].astype(int)

# Theo quy ước: irrelevant thì manual_label = 2.
gold.loc[gold["final_is_relevant"] == 0, "final_manual_label"] = 2
full.loc[full["is_relevant"] == 0, "manual_label"] = 2

match_info = match_gold_to_full(gold, full)

print("\nSố gold rows:", len(gold))
print("Số gold rows match được:", int((match_info["matched_full_count"] > 0).sum()))
print("Gold row matched rate:", round(float((match_info["matched_full_count"] > 0).mean()), 4))

# Kiểm tra match theo topic.
matched_topic = []
for _, r in match_info.iterrows():
    if pd.isna(r["matched_full_index"]):
        matched_topic.append(pd.NA)
    else:
        matched_topic.append(full.loc[int(r["matched_full_index"]), "topic"])

match_info["matched_full_topic"] = matched_topic

print("\nGold topic distribution:")
print(gold["topic"].value_counts().sort_index())

print("\nMatched full topic distribution:")
print(match_info["matched_full_topic"].value_counts().sort_index())

print("\nMatch method distribution:")
print(match_info["match_method"].value_counts())

unmatched = match_info[match_info["matched_full_count"] == 0]
print("\nSố gold rows chưa match được:", len(unmatched))
if len(unmatched) > 0:
    display(gold.loc[unmatched["gold_index"], ["id", "topic", "text"]].head(20))


Gold shape: (500, 12)
Full shape: (10718, 14)

Số gold rows: 500
Số gold rows match được: 500
Gold row matched rate: 1.0

Gold topic distribution:
topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Matched full topic distribution:
matched_full_topic
arsenal           100
iphone17series    100
ronaldo           100
s25series         100
vinfast_vf3       100
Name: count, dtype: int64

Match method distribution:
match_method
topic+video_id+created_at+text    500
Name: count, dtype: int64

Số gold rows chưa match được: 0


In [5]:
# ============================================================
# 3. TẠO FILE BENCHMARK GỌN
#    Giữ nguyên cột của gold set, chỉ thêm:
#    - ai_is_relevant
#    - ai_manual_label
#    - mismatch_status
# ============================================================

benchmark = gold.copy()

ai_is_relevant = []
ai_manual_label = []
mismatch_status = []

for _, r in match_info.iterrows():
    gold_idx = int(r["gold_index"])

    if pd.isna(r["matched_full_index"]):
        ai_is_relevant.append(pd.NA)
        ai_manual_label.append(pd.NA)
        mismatch_status.append("unmatched")
        continue

    full_idx = int(r["matched_full_index"])

    human_rel = int(gold.loc[gold_idx, "final_is_relevant"])
    human_sent = int(gold.loc[gold_idx, "final_manual_label"])

    ai_rel = int(full.loc[full_idx, "is_relevant"])
    ai_sent = int(full.loc[full_idx, "manual_label"])

    # Ép logic lại cho chắc.
    if ai_rel == 0:
        ai_sent = 2

    ai_is_relevant.append(ai_rel)
    ai_manual_label.append(ai_sent)

    relevance_wrong = human_rel != ai_rel
    sentiment_wrong = (human_rel == 1 and ai_rel == 1 and human_sent != ai_sent)

    if relevance_wrong and sentiment_wrong:
        status = "both_mismatch"
    elif relevance_wrong:
        status = "relevance_mismatch"
    elif sentiment_wrong:
        status = "manual_label_mismatch"
    else:
        status = "match"

    mismatch_status.append(status)

benchmark["ai_is_relevant"] = ai_is_relevant
benchmark["ai_manual_label"] = ai_manual_label
benchmark["mismatch_status"] = mismatch_status

benchmark.to_csv("gold_500_ai_benchmark_matched.csv", index=False, encoding="utf-8-sig")

print("Đã lưu file benchmark gọn: gold_500_ai_benchmark_matched.csv")
print("\nMismatch status:")
print(benchmark["mismatch_status"].value_counts(dropna=False))

display(benchmark.head())


Đã lưu file benchmark gọn: gold_500_ai_benchmark_matched.csv

Mismatch status:
mismatch_status
match                    322
relevance_mismatch       115
manual_label_mismatch     63
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
0,1,0qNL8VZHblA,arsenal,GÁY TO LÊN ANH EM PHÁO THỦ ƠIII,NaN,2026-05-21T11:50:36Z,YouTube,0qNL8VZHblA,0,0,1,0,1,0,match
1,2,0qNL8VZHblA,arsenal,It's not done,NaN,2026-05-20T14:05:33Z,YouTube,0qNL8VZHblA,0,0,0,2,0,2,match
2,3,0qNL8VZHblA,arsenal,Ai vào đây sau khi Arsenal đã vô địch ?,NaN,2026-05-20T05:21:01Z,YouTube,0qNL8VZHblA,0,0,1,2,1,2,match
3,4,0qNL8VZHblA,arsenal,0:21 0:22 MCT là số 1 arsenal tuổi 1:11 1:11,NaN,2026-05-18T11:42:01Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
4,5,0qNL8VZHblA,arsenal,"Giờ pháo lại sáng cửa rồi , ngày mai mà thắng ...",NaN,2026-05-09T00:38:20Z,YouTube,0qNL8VZHblA,0,0,1,0,0,2,relevance_mismatch


In [6]:
from sklearn.metrics import cohen_kappa_score
import pandas as pd

# ============================================================
# COHEN'S KAPPA GIỮA HUMAN GOLD SET VÀ AI LABEL
# ============================================================

# Nếu trong notebook đã có biến benchmark thì dùng luôn.
# Nếu chưa có, đọc từ file benchmark đã match đúng dòng.
try:
    kappa_df = benchmark.copy()
except NameError:
    kappa_df = pd.read_csv("gold_500_ai_benchmark_matched.csv")

print("========== COHEN'S KAPPA: HUMAN GOLD vs AI ==========")
print("Số dòng benchmark:", len(kappa_df))

# ============================================================
# 1. COHEN KAPPA CHO RELEVANCE
# ============================================================

rel_df = kappa_df.dropna(subset=["final_is_relevant", "ai_is_relevant"]).copy()

kappa_relevance = cohen_kappa_score(
    rel_df["final_is_relevant"],
    rel_df["ai_is_relevant"]
)

print("\n--- Relevance agreement ---")
print("Số mẫu:", len(rel_df))
print("Cohen's Kappa:", round(kappa_relevance, 4))

print("\nBảng đối chiếu relevance:")
print(
    pd.crosstab(
        rel_df["final_is_relevant"],
        rel_df["ai_is_relevant"],
        rownames=["Human final_is_relevant"],
        colnames=["AI is_relevant"]
    )
)

# ============================================================
# 2. COHEN KAPPA CHO SENTIMENT
# Chỉ tính trên các dòng mà cả Human và AI đều xác định là relevant.
# Đây là cách công bằng nhất để đánh giá sentiment riêng.
# ============================================================

sent_df = kappa_df[
    (kappa_df["final_is_relevant"] == 1) &
    (kappa_df["ai_is_relevant"] == 1)
].dropna(subset=["final_manual_label", "ai_manual_label"]).copy()

kappa_sentiment = cohen_kappa_score(
    sent_df["final_manual_label"],
    sent_df["ai_manual_label"]
)

print("\n--- Sentiment agreement, only Human relevant & AI relevant ---")
print("Số mẫu:", len(sent_df))
print("Cohen's Kappa:", round(kappa_sentiment, 4))

print("\nBảng đối chiếu sentiment:")
print(
    pd.crosstab(
        sent_df["final_manual_label"],
        sent_df["ai_manual_label"],
        rownames=["Human final_manual_label"],
        colnames=["AI manual_label"]
    )
)

# ============================================================
# 3. OPTIONAL: SENTIMENT KAPPA TRÊN TOÀN BỘ HUMAN RELEVANT
# Bao gồm cả trường hợp AI dự đoán irrelevant.
# Kết quả này thường thấp hơn và phản ánh lỗi relevance + sentiment gộp chung.
# ============================================================

sent_all_human_relevant = kappa_df[
    kappa_df["final_is_relevant"] == 1
].dropna(subset=["final_manual_label", "ai_manual_label"]).copy()

kappa_sentiment_all_human_relevant = cohen_kappa_score(
    sent_all_human_relevant["final_manual_label"],
    sent_all_human_relevant["ai_manual_label"]
)

print("\n--- Sentiment agreement, all Human relevant comments ---")
print("Số mẫu:", len(sent_all_human_relevant))
print("Cohen's Kappa:", round(kappa_sentiment_all_human_relevant, 4))

print("\nBảng đối chiếu sentiment trên toàn bộ Human relevant:")
print(
    pd.crosstab(
        sent_all_human_relevant["final_manual_label"],
        sent_all_human_relevant["ai_manual_label"],
        rownames=["Human final_manual_label"],
        colnames=["AI manual_label"]
    )
)

========== COHEN'S KAPPA: HUMAN GOLD vs AI ==========
Số dòng benchmark: 500

--- Relevance agreement ---
Số mẫu: 500
Cohen's Kappa: 0.5297

Bảng đối chiếu relevance:
AI is_relevant             0    1
Human final_is_relevant          
0                        155   64
1                         51  230

--- Sentiment agreement, only Human relevant & AI relevant ---
Số mẫu: 230
Cohen's Kappa: 0.5891

Bảng đối chiếu sentiment:
AI manual_label            0   1   2
Human final_manual_label            
0                         63   2   5
1                          9  47  19
2                         12  16  57

--- Sentiment agreement, all Human relevant comments ---
Số mẫu: 281
Cohen's Kappa: 0.4418

Bảng đối chiếu sentiment trên toàn bộ Human relevant:
AI manual_label            0   1   2
Human final_manual_label            
0                         63   2  16
1                          9  47  50
2                         12  16  66


In [7]:
# ============================================================
# 4. BENCHMARK RELEVANCE
# ============================================================

bench_rel = benchmark[benchmark["mismatch_status"] != "unmatched"].copy()

y_true_rel = bench_rel["final_is_relevant"].astype(int)
y_pred_rel = bench_rel["ai_is_relevant"].astype(int)

print("========== RELEVANCE BENCHMARK ==========")
print("Samples:", len(bench_rel))
print("Accuracy:", accuracy_score(y_true_rel, y_pred_rel))
print("Macro-F1:", f1_score(y_true_rel, y_pred_rel, average="macro"))

print("\nClassification report:")
print(classification_report(
    y_true_rel,
    y_pred_rel,
    labels=[0, 1],
    target_names=["human_irrelevant", "human_relevant"],
    digits=4
))

cm_rel = pd.DataFrame(
    confusion_matrix(y_true_rel, y_pred_rel, labels=[0, 1]),
    index=["Human irrelevant", "Human relevant"],
    columns=["AI irrelevant", "AI relevant"]
)

print("\nConfusion matrix:")
display(cm_rel)


========== RELEVANCE BENCHMARK ==========
Samples: 500
Accuracy: 0.77
Macro-F1: 0.7647058823529411

Classification report:
                  precision    recall  f1-score   support

human_irrelevant     0.7524    0.7078    0.7294       219
  human_relevant     0.7823    0.8185    0.8000       281

        accuracy                         0.7700       500
       macro avg     0.7674    0.7631    0.7647       500
    weighted avg     0.7692    0.7700    0.7691       500


Confusion matrix:


,AI irrelevant,AI relevant
Human irrelevant,155,64
Human relevant,51,230


In [8]:
# ============================================================
# 5. XUẤT FILE RELEVANCE MISMATCH GỌN
# ============================================================

relevance_mismatches = benchmark[
    (benchmark["mismatch_status"] != "unmatched")
    & (benchmark["final_is_relevant"].astype(int) != benchmark["ai_is_relevant"].astype(int))
].copy()

relevance_mismatches.to_csv(
    "relevance_mismatches_matched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: relevance_mismatches_matched.csv")
print("Số relevance mismatches:", len(relevance_mismatches))
print("\nTheo topic:")
print(relevance_mismatches["topic"].value_counts().sort_index())

display(relevance_mismatches.head(20))


Đã lưu: relevance_mismatches_matched.csv
Số relevance mismatches: 115

Theo topic:
topic
arsenal           17
iphone17series    20
ronaldo           23
s25series         21
vinfast_vf3       34
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
4,5,0qNL8VZHblA,arsenal,"Giờ pháo lại sáng cửa rồi , ngày mai mà thắng ...",NaN,2026-05-09T00:38:20Z,YouTube,0qNL8VZHblA,0,0,1,0,0,2,relevance_mismatch
7,8,0qNL8VZHblA,arsenal,Năm nay mà không vô địch được thì không còn gì...,NaN,2026-04-26T16:03:59Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
10,11,0qNL8VZHblA,arsenal,"pháo thủ cứ cuối mùa thì tịt ngòi, còn MC thì ...",NaN,2026-04-24T04:44:59Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
13,14,0qNL8VZHblA,arsenal,là một fan mc t thấy trận này pháo thủ chơi cũ...,NaN,2026-04-23T05:18:14Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
18,19,0qNL8VZHblA,arsenal,Ars thời Giáo Sư tôi yêu \nCòn bjo là màu xanh...,NaN,2026-04-22T01:53:47Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
22,23,0qNL8VZHblA,arsenal,ko vô địch mới nhớ mãi nhé 😅,NaN,2026-04-22T07:53:13Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
47,48,0qNL8VZHblA,arsenal,Asean hay nhưng tiếc la ko được may,NaN,2026-04-21T00:52:03Z,YouTube,0qNL8VZHblA,0,0,1,2,0,2,relevance_mismatch
48,49,0qNL8VZHblA,arsenal,"lời nguyền thu mơ ăn 6, đông ăn 4, xuân bắt đầ...",NaN,2026-04-21T00:27:14Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
60,61,0qNL8VZHblA,arsenal,Quá đen cho sơ nan,NaN,2026-04-20T16:01:33Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch
63,64,0qNL8VZHblA,arsenal,cúng 1 bàn còn thua thì chịu r😅😅,NaN,2026-04-20T15:35:40Z,YouTube,0qNL8VZHblA,0,0,1,1,0,2,relevance_mismatch


In [9]:
# ============================================================
# 6. BENCHMARK SENTIMENT / MANUAL_LABEL
#    Chỉ tính trên những dòng human relevant và AI cũng relevant.
# ============================================================

sent_bench = benchmark[
    (benchmark["mismatch_status"] != "unmatched")
    & (benchmark["final_is_relevant"].astype(int) == 1)
    & (benchmark["ai_is_relevant"].astype(int) == 1)
].copy()

y_true_sent = sent_bench["final_manual_label"].astype(int)
y_pred_sent = sent_bench["ai_manual_label"].astype(int)

print("========== SENTIMENT / MANUAL_LABEL BENCHMARK ==========")
print("Samples:", len(sent_bench))
print("Accuracy:", accuracy_score(y_true_sent, y_pred_sent))
print("Macro-F1:", f1_score(y_true_sent, y_pred_sent, average="macro"))

print("\nClassification report:")
print(classification_report(
    y_true_sent,
    y_pred_sent,
    labels=[0, 1, 2],
    target_names=["positive", "negative", "neutral"],
    digits=4
))

cm_sent = pd.DataFrame(
    confusion_matrix(y_true_sent, y_pred_sent, labels=[0, 1, 2]),
    index=["Human positive", "Human negative", "Human neutral"],
    columns=["AI positive", "AI negative", "AI neutral"]
)

print("\nConfusion matrix:")
display(cm_sent)


========== SENTIMENT / MANUAL_LABEL BENCHMARK ==========
Samples: 230
Accuracy: 0.7260869565217392
Macro-F1: 0.725452459187399

Classification report:
              precision    recall  f1-score   support

    positive     0.7500    0.9000    0.8182        70
    negative     0.7231    0.6267    0.6714        75
     neutral     0.7037    0.6706    0.6867        85

    accuracy                         0.7261       230
   macro avg     0.7256    0.7324    0.7255       230
weighted avg     0.7241    0.7261    0.7218       230


Confusion matrix:


,AI positive,AI negative,AI neutral
Human positive,63,2,5
Human negative,9,47,19
Human neutral,12,16,57


In [10]:
# ============================================================
# 7. XUẤT FILE MANUAL_LABEL MISMATCH GỌN
#    Chỉ xét trên những dòng human relevant và AI relevant.
# ============================================================

manual_label_mismatches = sent_bench[
    sent_bench["final_manual_label"].astype(int) != sent_bench["ai_manual_label"].astype(int)
].copy()

manual_label_mismatches.to_csv(
    "manual_label_mismatches_matched.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Đã lưu: manual_label_mismatches_matched.csv")
print("Số manual_label mismatches:", len(manual_label_mismatches))
print("\nTheo topic:")
print(manual_label_mismatches["topic"].value_counts().sort_index())

display(manual_label_mismatches.head(20))


Đã lưu: manual_label_mismatches_matched.csv
Số manual_label mismatches: 63

Theo topic:
topic
arsenal           21
iphone17series    13
ronaldo           10
s25series          7
vinfast_vf3       12
Name: count, dtype: int64


,id,video_id,topic,text,clean_text,created_at,source,source_item_id,like_count,reply_count,final_is_relevant,final_manual_label,ai_is_relevant,ai_manual_label,mismatch_status
3,4,0qNL8VZHblA,arsenal,0:21 0:22 MCT là số 1 arsenal tuổi 1:11 1:11,NaN,2026-05-18T11:42:01Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
9,10,0qNL8VZHblA,arsenal,Lách qua khe cửa hẹp pháo vẫn quyết định được ...,NaN,2026-04-25T10:46:46Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
12,13,0qNL8VZHblA,arsenal,@hoanglonglephan4020 9 ngày trước fan Ars cười...,NaN,2026-04-23T09:32:25Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
36,37,0qNL8VZHblA,arsenal,A phú thảo trận này hơi đen,NaN,2026-04-21T10:09:12Z,YouTube,0qNL8VZHblA,0,0,1,2,1,1,manual_label_mismatch
39,40,0qNL8VZHblA,arsenal,"Cặp CB của Mci uy tín quá, nếu ko có sai lầm c...",NaN,2026-04-21T05:01:21Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
42,43,0qNL8VZHblA,arsenal,Chuông kì bu thì đá mấy vẫn vậy th à = )))))))),NaN,2026-04-21T04:30:28Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
55,56,0qNL8VZHblA,arsenal,Kẹt pháo r kẹt pháo r😂,NaN,2026-04-20T17:36:42Z,YouTube,0qNL8VZHblA,0,0,1,1,1,2,manual_label_mismatch
56,57,0qNL8VZHblA,arsenal,Anh em arsenal điểm danh nào 😂😂😂,NaN,2026-04-20T17:30:51Z,YouTube,0qNL8VZHblA,0,0,1,2,1,0,manual_label_mismatch
57,58,0qNL8VZHblA,arsenal,"etita là người rất giỏi, giỏi trong những trận...",NaN,2026-04-20T17:24:07Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
61,62,0qNL8VZHblA,arsenal,phấn đào nguuu,NaN,2026-04-20T15:56:12Z,YouTube,0qNL8VZHblA,0,0,1,1,1,0,manual_label_mismatch
